# ✈️ US Airline Performance Analysis
**Dataset:** Bureau of Transportation Statistics — 2018 & 2019  
**Records:** 14.6M+ flights  
**Goal:** Aggregate raw flight data into clean tables for Looker Studio dashboard

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os

# Locate data files on Drive
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if '2018' in f or '2019' in f:
            print(os.path.join(root, f))

/content/drive/MyDrive/2018.csv
/content/drive/MyDrive/2019.csv
/content/drive/MyDrive/Уник/SCAN_20180609_123618763.pdf


In [3]:
import pandas as pd

# Preview column names before loading full dataset
df_peek = pd.read_csv('/content/drive/MyDrive/2018.csv', nrows=3)
print(df_peek.columns.tolist())

['FL_DATE', 'OP_CARRIER', 'OP_CARRIER_FL_NUM', 'ORIGIN', 'DEST', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'WHEELS_ON', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME', 'ARR_DELAY', 'CANCELLED', 'CANCELLATION_CODE', 'DIVERTED', 'CRS_ELAPSED_TIME', 'ACTUAL_ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY', 'Unnamed: 27']


In [4]:
import pandas as pd

def load_file(path):
    """Load CSV with only required columns to save memory."""
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()
    return df[[c for c in [
        'FL_DATE', 'OP_CARRIER', 'OP_UNIQUE_CARRIER',
        'ORIGIN', 'DEST', 'DEP_DELAY', 'ARR_DELAY', 'CANCELLED',
        'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
        'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'
    ] if c in df.columns]]

df18 = load_file('/content/drive/MyDrive/2018.csv')
df19 = load_file('/content/drive/MyDrive/2019.csv')

# Unify carrier column name across both files
for d in [df18, df19]:
    if 'OP_UNIQUE_CARRIER' in d.columns and 'OP_CARRIER' not in d.columns:
        d.rename(columns={'OP_UNIQUE_CARRIER': 'OP_CARRIER'}, inplace=True)

df = pd.concat([df18, df19], ignore_index=True)

# Extract date features from FL_DATE
df['FL_DATE']     = pd.to_datetime(df['FL_DATE'])
df['YEAR']        = df['FL_DATE'].dt.year
df['MONTH']       = df['FL_DATE'].dt.month
df['DAY_OF_WEEK'] = df['FL_DATE'].dt.dayofweek  # 0=Mon, 6=Sun

print(f"Rows loaded: {len(df):,}")
print(df.dtypes)

Rows loaded: 14,635,483
FL_DATE                datetime64[ns]
OP_CARRIER                     object
ORIGIN                         object
DEST                           object
DEP_DELAY                     float64
ARR_DELAY                     float64
CANCELLED                     float64
CARRIER_DELAY                 float64
WEATHER_DELAY                 float64
NAS_DELAY                     float64
SECURITY_DELAY                float64
LATE_AIRCRAFT_DELAY           float64
YEAR                            int32
MONTH                           int32
DAY_OF_WEEK                     int32
dtype: object


In [5]:
# ── Aggregation ───────────────────────────────────────────────────────────────

# 1. KPI summary (single-row overview)
delayed = df[df['DEP_DELAY'] > 15]
kpi = pd.DataFrame([{
    'total_flights':    len(df),
    'ontime_rate_pct':  round((1 - len(delayed) / len(df)) * 100, 1),
    'avg_delay_min':    round(df[df['DEP_DELAY'] > 0]['DEP_DELAY'].mean(), 1),
    'cancellation_pct': round(df['CANCELLED'].sum() / len(df) * 100, 2)
}])

# 2. Monthly performance (24 rows: 12 months x 2 years)
monthly = (
    df.groupby(['YEAR', 'MONTH'])
    .agg(
        total_flights = ('FL_DATE', 'count'),
        avg_delay     = ('DEP_DELAY', 'mean'),
        cancellations = ('CANCELLED', 'sum')
    )
    .reset_index()
)
monthly['ontime_pct'] = round((1 - monthly['cancellations'] / monthly['total_flights']) * 100, 1)
monthly['avg_delay']  = monthly['avg_delay'].round(1)

# 3. Airlines ranking — sorted by avg delay descending
airlines = (
    df.groupby('OP_CARRIER')
    .agg(
        total_flights = ('FL_DATE', 'count'),
        avg_delay     = ('DEP_DELAY', 'mean'),
        cancellations = ('CANCELLED', 'sum')
    )
    .reset_index()
)
airlines['ontime_pct']       = round((1 - airlines['cancellations'] / airlines['total_flights']) * 100, 1)
airlines['avg_delay']        = airlines['avg_delay'].round(1)
airlines['cancellation_pct'] = round(airlines['cancellations'] / airlines['total_flights'] * 100, 2)
airlines = airlines.sort_values('avg_delay', ascending=False)

# 4. Delay causes breakdown
causes = pd.DataFrame({
    'cause': ['Late aircraft', 'Carrier', 'NAS', 'Weather', 'Security'],
    'avg_minutes': [
        df['LATE_AIRCRAFT_DELAY'].mean().round(1),
        df['CARRIER_DELAY'].mean().round(1),
        df['NAS_DELAY'].mean().round(1),
        df['WEATHER_DELAY'].mean().round(1),
        df['SECURITY_DELAY'].mean().round(1)
    ]
})

# 5. Worst routes — popular routes only (500+ flights), top 20 by avg delay
routes = (
    df.groupby(['ORIGIN', 'DEST'])
    .agg(
        total_flights = ('FL_DATE', 'count'),
        avg_delay     = ('DEP_DELAY', 'mean')
    )
    .reset_index()
)
routes = routes[routes['total_flights'] > 500]
routes['avg_delay'] = routes['avg_delay'].round(1)
routes = routes.sort_values('avg_delay', ascending=False).head(20)

# ── Export to Google Drive ────────────────────────────────────────────────────
kpi.to_csv('/content/drive/MyDrive/kpi_summary.csv', index=False)
monthly.to_csv('/content/drive/MyDrive/monthly_performance.csv', index=False)
airlines.to_csv('/content/drive/MyDrive/airlines_ranking.csv', index=False)
causes.to_csv('/content/drive/MyDrive/delay_causes.csv', index=False)
routes.to_csv('/content/drive/MyDrive/worst_routes.csv', index=False)

print("Done! 5 aggregated files saved to Google Drive.")
print(f"\nKPI Summary:\n{kpi}")
print(f"\nTop 5 airlines by punctuality:\n{airlines.head()}")

Done! 5 aggregated files saved to Google Drive.

KPI Summary:
   total_flights  ontime_rate_pct  avg_delay_min  cancellation_pct
0       14635483             82.4           39.5               0.8

Top 5 airlines by punctuality:
   OP_CARRIER  total_flights  avg_delay  cancellations  ontime_pct  \
6          F9         255578       17.0         2328.0        99.1   
3          B6         602421       16.8         6419.0        98.9   
5          EV         337573       14.3         5670.0        98.3   
16         YV         443026       12.5         5530.0        98.8   
13         UA        1247475       11.5         4903.0        99.6   

    cancellation_pct  
6               0.91  
3               1.07  
5               1.68  
16              1.25  
13              0.39  
